# Apply Work-Author Curations

Applies `add` and `remove` work-author curations from `openalex.works.work_author_add_curations` and `openalex.works.work_author_remove_curations` onto `openalex.works.work_authors`. Runs after `Author_Matching` (which finalizes `author_id`) and before `UpdateWorkAuthorships` (which materializes the final authorships struct).

## Semantics

- **`add(B, W, k)` = transfer**: rewrite `work_authors.author_id` to B at the slot the curator clicked (`k = author_sequence`). Apply is positional MERGE only — `(work_id, author_sequence)` is the key. Curations land within ~24h of submission, so re-ingest drift is not a real concern at apply time.
- **`remove(A, W)` = sticky disclaim**: NULL `work_authors.author_id` at any slot of W where it currently equals A. Next `MatchAuthors` cycle re-picks; the block-list filter in `MatchAuthors.blocked_candidates` (reading `work_author_remove_curations` directly) prevents A from being re-selected.

## Captured `raw_author_name`

`work_author_add_curations` persists `raw_author_name` from `work_authors @ (work_id, author_sequence)` at sync time, but the apply step does NOT use it as a guard — it's diagnostic only. Preserved for later audits / re-match detection if a re-ingest reorders slots after the curation was applied.

## Affected works queue

Every work_id touched (added or removed) is inserted into `openalex.institutions.curated_work_ids_pending_sync` so `UpdateWorkAuthorships` re-flows them on its next run.

## Apply add curations to work_authors

Positional MERGE on `(work_id, author_sequence)` directly from `work_author_add_curations`. Skips the UPDATE when the slot already holds the curated author (no-op churn protection).

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING openalex.works.work_author_add_curations AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
WHEN MATCHED AND (target.author_id IS NULL OR target.author_id <> source.author_id) THEN
  UPDATE SET
    target.author_id  = source.author_id,
    target.updated_at = current_timestamp()

## Apply remove curations to work_authors

NULL the `author_id` at any (work_id, author_sequence) where it currently equals the disclaimed author. Non-positional — every matching slot for the (work_id, author_id) pair gets NULL'd. Next `MatchAuthors` cycle re-picks; the block-list filter in `MatchAuthors.blocked_candidates` prevents A from being re-selected on subsequent runs.

In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING openalex.works.work_author_remove_curations AS source
ON target.work_id = source.work_id
   AND target.author_id = source.author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id  = NULL,
    target.updated_at = current_timestamp()

## Queue affected work_ids for re-sync

Insert every work_id touched by add or remove curations into `curated_work_ids_pending_sync`. `UpdateWorkAuthorships` picks them up (via its `WHERE id IN (SELECT work_id FROM curated_work_ids_pending_sync)` clause) and truncates the queue at the end of its run.

In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT DISTINCT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM (
    SELECT work_id FROM openalex.works.work_author_add_curations
    UNION
    SELECT work_id FROM openalex.works.work_author_remove_curations
)

## Verify

In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM openalex.works.work_author_add_curations)    AS add_curations_total,
    (SELECT COUNT(*) FROM openalex.works.work_author_remove_curations) AS remove_curations_total

In [ ]:
%sql
SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name, wa.updated_at
FROM openalex.works.work_authors wa
WHERE wa.work_id IN (
    SELECT work_id FROM openalex.works.work_author_add_curations
    UNION
    SELECT work_id FROM openalex.works.work_author_remove_curations
)
ORDER BY wa.updated_at DESC
LIMIT 100